# Notebook 3 — Graph DB + ReAct Agent Testing

Tests the full **Reasoning Layer** end-to-end.

| # | What it tests |
|---|---------------|
| 1 | `GraphClient` — insert typed triples, query by Cypher |
| 2 | Agent tools — `search_semantic_events`, `query_graph_relationships`, `verify_physics_math` |
| 3 | Embedding-based intent router |
| 4 | Full **ReAct agent** (requires Ollama + `qwen2.5:72b`) |

> All 3 databases are seeded with synthetic data before agent tests.

In [ ]:
import sys, os, json, shutil
sys.path.insert(0, os.path.abspath('..'))
print('sys.path ready')

## 1. GraphClient — Typed node insert + Cypher queries

In [ ]:
from src.memory_layer.graph_client import GraphClient

TEST_GRAPH = '/tmp/test_graph_db'
if os.path.exists(TEST_GRAPH):
    shutil.rmtree(TEST_GRAPH)

gc = GraphClient(db_path=TEST_GRAPH)

triples = [
    {'subject': 'Vehicle 4',   'subject_type': 'Vehicle',       'predicate': 'tailgating',
     'object': 'Vehicle 9',    'object_type': 'Vehicle',         'timestamp': 12.5},
    {'subject': 'Vehicle 4',   'subject_type': 'Vehicle',       'predicate': 'braking_hard',
     'object': 'intersection', 'object_type': 'Infrastructure',  'timestamp': 13.0},
    {'subject': 'Pedestrian 1','subject_type': 'Pedestrian',    'predicate': 'crossing_in_front_of',
     'object': 'Vehicle 3',    'object_type': 'Vehicle',         'timestamp': 14.0},
    {'subject': 'Vehicle 6',   'subject_type': 'Vehicle',       'predicate': 'running_red_light',
     'object': 'traffic_light','object_type': 'Infrastructure',  'timestamp': 18.0},
]
gc.insert_vlm_triples(triples, time_window='10.0-20.0')
print(f'Inserted {len(triples)} typed triples')

In [ ]:
# All interactions in the time window
q1 = "MATCH (s)-[r:INTERACTS_WITH]->(o) WHERE r.trajectory_time_window = '10.0-20.0' RETURN s.name, r.predicate, o.name"
r1 = gc.query_graph(q1)
print("All interactions in 10.0-20.0:")
print(json.dumps(r1, indent=2))

# Vehicle -> Infrastructure only
q2 = "MATCH (s:Vehicle)-[r:INTERACTS_WITH]->(o:Infrastructure) RETURN s.name, r.predicate, o.name"
r2 = gc.query_graph(q2)
print("\nVehicle -> Infrastructure:")
print(json.dumps(r2, indent=2))

# Pedestrian interactions
q3 = "MATCH (s:Pedestrian)-[r:INTERACTS_WITH]->(o) RETURN s.name, r.predicate, o.name"
r3 = gc.query_graph(q3)
print("\nPedestrian interactions:")
print(json.dumps(r3, indent=2))

assert len(r1) == 4, f'Expected 4, got {len(r1)}'
assert len(r2) == 2, f'Expected 2 V->I edges, got {len(r2)}'
assert len(r3) == 1, f'Expected 1 pedestrian edge, got {len(r3)}'
print("\nPASS: GraphClient Cypher queries return correct typed results")
gc.close()

## 2. Seed all 3 DBs and test each agent tool

In [ ]:
import numpy as np
from src.physics_engine.kinematics import KinematicEstimator
from src.memory_layer.duckdb_client import DuckDBClient
from src.memory_layer.milvus_client import SemanticVectorStore

FPS = 30.0
DT  = 1.0 / FPS

DUCK_PATH   = '/tmp/agent_duck.duckdb'
MILVUS_PATH = '/tmp/agent_milvus.db'
GRAPH_PATH  = '/tmp/agent_graph'

for p in [DUCK_PATH, MILVUS_PATH]:
    if os.path.exists(p): os.remove(p)
if os.path.exists(GRAPH_PATH): shutil.rmtree(GRAPH_PATH)

# Seed DuckDB: 60 frames, 2 vehicles
duck = DuckDBClient(db_path=DUCK_PATH)
kin  = KinematicEstimator(fps=FPS)
for frame_id in range(60):
    ts     = frame_id * DT
    coords = {4: (0.5 * 3.0 * ts**2, 5.0), 9: (40.0 - 0.5 * 2.0 * ts**2, 8.0)}
    duck.insert_state_vectors(ts, frame_id, kin.update(coords))
duck.close()
print('DuckDB seeded')

# Seed Milvus
milvus = SemanticVectorStore(db_path=MILVUS_PATH)
for desc, ts, te in [
    ('Vehicle 4 tailgating Vehicle 9 at high speed, collision risk.', 10.0, 20.0),
    ('Vehicle 2 brakes hard at intersection; near-miss with Vehicle 7.', 10.0, 20.0),
    ('Pedestrian crossing in front of Vehicle 3 on the zebra crossing.', 10.0, 20.0),
    ('Vehicle 6 running a red light at the traffic light.', 10.0, 20.0),
]:
    milvus.insert_event_chunk(desc, ts, te)
milvus.close()
print('Milvus seeded')

# Seed Graph
gc2 = GraphClient(db_path=GRAPH_PATH)
gc2.insert_vlm_triples([
    {'subject': 'Vehicle 4', 'subject_type': 'Vehicle', 'predicate': 'tailgating',
     'object': 'Vehicle 9', 'object_type': 'Vehicle', 'timestamp': 12.5},
    {'subject': 'Vehicle 4', 'subject_type': 'Vehicle', 'predicate': 'braking_hard',
     'object': 'intersection', 'object_type': 'Infrastructure', 'timestamp': 13.0},
], time_window='10.0-20.0')
gc2.close()
print('Graph seeded')
print('\nAll 3 databases ready for agent testing.')

In [ ]:
import src.agentic_orchestrator.tools as tools_mod

# Inject our pre-seeded DB singletons into the tools module
tools_mod._milvus_db = SemanticVectorStore(db_path=MILVUS_PATH)
tools_mod._graph_db  = GraphClient(db_path=GRAPH_PATH)
tools_mod._duckdb_db = DuckDBClient(db_path=DUCK_PATH)

from src.agentic_orchestrator.tools import (
    search_semantic_events,
    query_graph_relationships,
    verify_physics_math
)

# --- Tool 1: Milvus semantic search ---
print("=== Tool 1: search_semantic_events ===")
r1 = search_semantic_events.invoke({'query': 'vehicle tailgating at high speed'})
print(r1)
assert 'tailgating' in r1.lower() or 'Vehicle' in r1

# --- Tool 2: Graph query ---
print("\n=== Tool 2: query_graph_relationships ===")
cypher = "MATCH (s)-[r:INTERACTS_WITH]->(o) WHERE r.trajectory_time_window = '10.0-20.0' RETURN s.name, r.predicate, o.name"
r2 = query_graph_relationships.invoke({'cypher_query': cypher})
print(r2)
data2 = json.loads(r2)
assert len(data2) >= 1

# --- Tool 3: Physics verification ---
print("\n=== Tool 3: verify_physics_math ===")
r3 = verify_physics_math.invoke({'start_time': 0.3, 'end_time': 1.5, 'track_id': 4})
print(r3)
data3 = json.loads(r3)
assert 'max_speed_meters_per_second' in data3

print("\nPASS: All 3 agent tools return structured results")

## 3. Embedding-based Intent Router

Verify `all-MiniLM-L6-v2` classifies physics vs semantic queries correctly.

In [ ]:
from src.agentic_orchestrator.hierarchical_router import _classify_intent

physics_queries = [
    'How fast was Vehicle 4 going at the time of impact?',
    'Did Vehicle 9 brake too hard?',
    'What was the deceleration of Vehicle 3?',
]
semantic_queries = [
    'What events happened near the intersection?',
    'Were there any near-misses in the video?',
    'Describe the traffic scene at 10 seconds.',
]

print('Physics queries (expect: full_analysis):')
for q in physics_queries:
    intent = _classify_intent(q)
    tag = 'OK' if intent == 'full_analysis' else 'MISMATCH'
    print(f'  [{tag}] {intent:<20} <- {q}')

print('\nSemantic queries (expect: semantic_lookup):')
for q in semantic_queries:
    intent = _classify_intent(q)
    tag = 'OK' if intent == 'semantic_lookup' else 'MISMATCH'
    print(f'  [{tag}] {intent:<20} <- {q}')

assert all(_classify_intent(q) == 'full_analysis'   for q in physics_queries)
assert all(_classify_intent(q) == 'semantic_lookup' for q in semantic_queries)
print('\nPASS: Embedding router classifies all 6 queries correctly')

## 4. Full ReAct Agent

> Requires Ollama running with `qwen2.5:72b`. Run `ollama pull qwen2.5:72b` if not present.

In [ ]:
import subprocess
check = subprocess.run(['ollama', 'list'], capture_output=True, text=True)

if 'qwen2.5:72b' not in check.stdout:
    print('WARNING: qwen2.5:72b not available. Skipping agent test.')
    print('Pull it with: ollama pull qwen2.5:72b')
else:
    from src.agentic_orchestrator.sequential_pipeline import agent_app

    test_queries = [
        'Were there any near-misses or dangerous events in the video?',
        'What was Vehicle 4 doing, and which vehicles did it interact with?',
        'Did Vehicle 4 brake hard? What was its maximum speed?',
    ]

    for query in test_queries:
        sep = '=' * 60
        print(f'\n{sep}')
        print(f'QUERY: {query}')
        print(sep)
        state = agent_app.invoke({'query': query})
        print(f"ANSWER: {state.get('final_summary', 'No answer generated.')}")

    print('\nPASS: ReAct agent completed all 3 queries')